# Notebook 1: Data Preparation & EDA
Thực thi trên Kaggle: Cài đặt các thư viện cần thiết bằng lệnh dưới

In [ ]:
!pip install datasets pandas matplotlib seaborn transformers

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer

## 1. Tải dữ liệu và Lấy mẫu (Sampling)
Dùng bộ `nam194/vietnews`.
Ở đây ta sẽ áp dụng Data-centric AI: Lọc bỏ các bài báo có hiện tượng "Lead-Bias"
(phần tóm tắt sao chép y nguyên 3 câu đầu của bài báo). Nếu đưa dữ liệu này vào train,
mô hình sẽ lười biếng và chỉ học cách copy đoạn đầu.

In [ ]:
# Tải dataset
print("Loading dataset...")
dataset = load_dataset("nam194/vietnews", split="train")

# Shuffle để đảm bảo tính ngẫu nhiên
dataset = dataset.shuffle(seed=42)

def is_not_lead_biased(example):
    # Lấy 3 câu đầu của bài báo
    sentences = str(example['article']).split('.')
    lead_3 = '.'.join(sentences[:3])
    
    # Dùng phương pháp Word Overlap (Jaccard-like) thay vì ROUGE để chạy cực nhanh (1 giây cho 30k mẫu thay vì 30 phút)
    lead_words = set(lead_3.lower().split())
    abstract_words = set(str(example['abstract']).lower().split())
    
    if not abstract_words or not lead_words:
        return False
        
    overlap_ratio = len(abstract_words.intersection(lead_words)) / len(abstract_words)
    
    # Chỉ GIỮ LẠI các bài báo có tỷ lệ trùng lặp từ vựng < 80% (tức là không copy 100% từ 3 câu đầu)
    return overlap_ratio < 0.8

print("Đang lọc dữ liệu Lead-Bias (Data-centric)... Quá trình này rất nhanh nhờ Word Overlap.")
# Lọc trên một lượng lớn mẫu (ví dụ 30,000 mẫu đầu tiên) để lấy ra đủ 12,000 mẫu sạch
subset_to_filter = dataset.select(range(30000))
filtered_dataset = subset_to_filter.filter(is_not_lead_biased, num_proc=4)

print(f"Số lượng mẫu sạch sau khi lọc: {len(filtered_dataset)}")

## 2. Chia tập Train/Val/Test
Đảm bảo lấy CHÍNH XÁC 12,000 mẫu sạch để chia 10k/1k/1k.

In [ ]:
# Lấy đúng 12000 mẫu đầu tiên từ tập đã lọc
final_subset = filtered_dataset.select(range(12000))

# Chia tập train/val/test
train_dataset = final_subset.select(range(0, 10000))
val_dataset = final_subset.select(range(10000, 11000))
test_dataset = final_subset.select(range(11000, 12000))

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

## 3. EDA (Exploratory Data Analysis)
Đo đạc phân bố chiều dài token sử dụng tokenizer của BARTpho-syllable.
LƯU Ý: Ta sử dụng `vinai/bartpho-syllable` thay vì bản word-level 
để tránh phải cài đặt `VnCoreNLP` phức tạp trên Kaggle, model vẫn đảm bảo độ chính xác cực tốt.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/bartpho-syllable")

def get_lengths(example):
    return {
        "article_len": len(tokenizer(example["article"], truncation=False)["input_ids"]),
        "summary_len": len(tokenizer(example["abstract"], truncation=False)["input_ids"])
    }

print("Đang tính toán chiều dài token...")
eda_dataset = train_dataset.map(get_lengths, num_proc=4)

# Chuyển sang pandas để vẽ biểu đồ
df = eda_dataset.to_pandas()

In [ ]:
# Vẽ biểu đồ phân bố
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df["article_len"], bins=50, color='blue', kde=True)
plt.title("Phân bố chiều dài Article (Tokens)")
plt.xlabel("Số lượng tokens")

plt.subplot(1, 2, 2)
sns.histplot(df["summary_len"], bins=50, color='orange', kde=True)
plt.title("Phân bố chiều dài Summary (Tokens)")
plt.xlabel("Số lượng tokens")

plt.tight_layout()
plt.show()

## 4. Lưu dữ liệu
Lưu ra file CSV để sử dụng ở các Notebook sau.

In [ ]:
train_dataset.to_pandas().to_csv("train_10k.csv", index=False)
val_dataset.to_pandas().to_csv("val_1k.csv", index=False)
test_dataset.to_pandas().to_csv("test_1k.csv", index=False)
print("Đã lưu các file train_10k.csv, val_1k.csv, test_1k.csv thành công!")